In [1]:
import numpy as np
import scipy


import jax 
import jax.numpy as jnp

In [2]:
jax.devices()

[CpuDevice(id=0)]

In [4]:
a = np.random.normal(size=(500, 500))

In [5]:
b = a @ a.T

In [6]:
b32 = b.astype('float32')

In [7]:
sqrt_np = scipy.linalg.sqrtm(b32)

In [8]:
def sqrtm_newton_schulz(a):
    k = 10
    normalization = np.trace(a)
    y = a.copy() / normalization
    z = np.eye(a.shape[0])
    identity = np.eye(a.shape[0])
    for i in range(k):
        y_now = 0.5 * y @ (3. * identity - z @ y)
        z_now = 0.5 * (3. * identity - z @ y) @ z
        y = y_now
        z = z_now
    return y * np.sqrt(normalization)

In [9]:
sqrt_ns = sqrtm_newton_schulz(b32)

In [10]:
np.mean(sqrt_np.ravel())

np.float32(0.03874253)

In [11]:
np.mean((sqrt_ns-sqrt_np).ravel())

np.float64(-0.00119571728378204)

In [12]:
%%timeit

sqrt_np = scipy.linalg.sqrtm(b32)

76.9 ms ± 2.69 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [13]:
%%timeit

sqrt_ns = sqrtm_newton_schulz(b32)

146 ms ± 4.93 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [14]:
jax.devices()

[CpuDevice(id=0)]

In [15]:
b32_j = jnp.array(b32)

In [16]:
@jax.jit
def sqrtm_newton_schulz_jax(a):
    k = 10
    normalization = jnp.trace(a)
    y = a.copy() / normalization
    z = jnp.eye(a.shape[0])
    identity = jnp.eye(a.shape[0])
    for i in range(k):
        y_now = 0.5 * y @ (3. * identity - z @ y)
        z_now = 0.5 * (3. * identity - z @ y) @ z
        y = y_now
        z = z_now
    return y * jnp.sqrt(normalization)

In [17]:
sqrt_ns_j = sqrtm_newton_schulz(b32_j)

In [18]:
np.mean((sqrt_ns_j-sqrt_np).ravel())

Array(-0.00119567, dtype=float32)

In [19]:
%%timeit

sqrt_ns = sqrtm_newton_schulz_jax(b32_j)

23.1 ms ± 862 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [20]:
@jax.jit
def sqrtm_newton_schulz_jax_loop(a):

    def body_fun(i, pars):
        y, z = pars
        y_now = 0.5 * y @ (3. * identity - z @ y)
        z_now = 0.5 * (3. * identity - z @ y) @ z
        return (y_now, z_now)
    k = 10
    normalization = jnp.trace(a)
    y = a.copy() / normalization
    z = jnp.eye(a.shape[0])
    identity = jnp.eye(a.shape[0])
    (y, z) = jax.lax.fori_loop(0, k, body_fun, (y, z))
    return y * jnp.sqrt(normalization)

In [21]:
%%timeit
for i in range(100):
    sqrt_ns = sqrtm_newton_schulz_jax_loop(b32_j)

2.59 s ± 72 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [22]:
sqrt_np

array([[19.052967  , -0.50858086, -1.0219964 , ..., -0.19538696,
        -0.7552064 , -0.525272  ],
       [-0.5085845 , 18.608004  , -0.16867056, ...,  0.3069748 ,
        -0.08063506, -0.53959966],
       [-1.0219979 , -0.16866836, 19.206844  , ...,  0.59131545,
        -1.1983159 ,  0.08492143],
       ...,
       [-0.19538441,  0.30697063,  0.5913188 , ..., 20.38177   ,
        -0.04884855, -0.428203  ],
       [-0.75520504, -0.08063443, -1.1983167 , ..., -0.04885051,
        18.41876   , -0.22511531],
       [-0.52527267, -0.5396046 ,  0.08491968, ..., -0.42820197,
        -0.22511937, 18.25245   ]], shape=(500, 500), dtype=float32)

In [23]:
sqrt_ns_j

Array([[18.283699  , -0.54130924, -1.0946333 , ..., -0.21947311,
        -0.81248313, -0.51957875],
       [-0.5413098 , 17.932552  , -0.19258657, ...,  0.32435703,
        -0.09977297, -0.5952336 ],
       [-1.0946301 , -0.19258729, 18.51881   , ...,  0.6044535 ,
        -1.3065561 ,  0.06467051],
       ...,
       [-0.21947227,  0.3243581 ,  0.60445255, ..., 19.808167  ,
        -0.06976216, -0.41021097],
       [-0.8124861 , -0.09977258, -1.306556  , ..., -0.0697618 ,
        17.714638  , -0.27328008],
       [-0.51957875, -0.59523517,  0.06466959, ..., -0.41021082,
        -0.27327967, 17.566288  ]], dtype=float32)

In [24]:
jax.devices('gpu')

RuntimeError: Unknown backend: 'gpu' requested, but no platforms that are instances of gpu are present. Platforms present are: cpu